# ELENDRIL 004T4 â€” Expanded two-hop comparison

Use a Colab T4 GPU runtime. This is an owner-run development experiment. The notebook first qualifies a one-device canary, then runs the expanded comparison only when `EXECUTE_CAMPAIGN` is explicitly enabled. Allow up to 75 minutes total for the campaign in one session. Keep the same run ID to resume from verified Google Drive checkpoints.

The four arms use the same 128-wide, four-layer core, paired examples, batch size 16 and 1,536 updates at seeds 7, 19 and 41. The arms are dense residual capacity control, contextual pointer-generator, one-read hybrid and two-read hybrid. Every 384 updates a resumable checkpoint and raw scoring receipt are copied to Drive and verified. The comparison matches updates and examples. Assess cost separately using actual training seconds and collected memory telemetry. Development gates are those of ELD-DEV-004. The larger profile is a separate experiment from the local 42k-parameter study.

The notebook expects the repository to be cloneable. If access requires authentication, prepare an authorized checkout and set `REPO` to its path before running the bootstrap. Keep authentication outside the notebook. [Colab runtime FAQ](https://research.google.com/colaboratory/faq.html) Â· [PyTorch wheel catalogue](https://pytorch.org/get-started/previous-versions/)

In [ ]:
from pathlib import Path
import os, re, subprocess, sys, time, json, hashlib, shutil, tempfile

EXECUTE_CAMPAIGN = False
RUN_ID = "eld-dev-004t4-v1"
SOURCE = "cc38d9850053941097f1d80a4c0b3bbd57f765e7"
REPO = Path("/content/elendril-004-source")
LOCAL = Path("/content/elendril-004t4") / RUN_ID
MOUNT = Path("/content/drive")
DRIVE = MOUNT / "MyDrive" / "ELENDRIL" / RUN_ID
assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", RUN_ID)
assert "torch" not in sys.modules, "Restart the Colab runtime before installing the pinned wheel."

def run(command, timeout=120, **kwargs):
    return subprocess.run(command, check=True, timeout=timeout, **kwargs)

gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=index,name,driver_version,memory.total,memory.free",
     "--format=csv,noheader,nounits"], text=True).strip().splitlines()
assert len(gpu) == 1 and "T4" in gpu[0], gpu
assert int(gpu[0].split(",")[-1]) >= 8000, gpu
if not REPO.exists():
    run(["git","clone","--no-checkout","--single-branch","--branch","Elendril",
         "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git",str(REPO)], timeout=300)
else:
    assert not subprocess.check_output(["git","status","--porcelain"],cwd=REPO,text=True).strip()
run(["git","checkout","--detach",SOURCE],cwd=REPO)
assert subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip() == SOURCE
run([sys.executable,"-m","pip","install","torch==2.8.0","--index-url",
     "https://download.pytorch.org/whl/cu126"], timeout=900)
os.environ.update(OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", CUBLAS_WORKSPACE_CONFIG=":4096:8")
sys.path.insert(0,str(REPO))
LOCAL.mkdir(parents=True,exist_ok=True)
print(gpu[0])

In [ ]:
import torch
from v5_training.checkpoint import CheckpointStore
from v5_training.persistent_store import materialize_local, mirror_checkpoint, read_mirror_pointer

assert str(torch.__version__).startswith("2.8.0+cu126")
assert torch.cuda.is_available() and torch.cuda.device_count() == 1
MODULE = [sys.executable, "-m", "anra_v5.elendril"]

def pilot(method, seed, total, until, root, output, *, profile, device, resume=False, timeout=600):
    command = [*MODULE,"pilot","--surface","join-v1","--arm","binding","--method",method,
               "--seed",str(seed),"--steps",str(total),"--until",str(until),"--profile",profile,
               "--device",device,"--root",str(root),"--output",str(output)]
    if resume:
        command.append("--resume")
    with output.with_suffix(".log").open("xb") as log:
        run(command, cwd=REPO, timeout=timeout, stdout=log, stderr=subprocess.STDOUT)
    run([*MODULE,"rescore","--receipt",str(output)], cwd=REPO, timeout=120)
    return json.loads(output.read_bytes())

with tempfile.TemporaryDirectory(prefix="elendril-canary-",dir=LOCAL) as temp:
    temp = Path(temp)
    args = ("two-read",7,32)
    cpu = pilot(*args,2,temp/"cpu",temp/"cpu.json",profile="micro",device="cpu")
    direct = pilot(*args,2,temp/"direct",temp/"direct.json",profile="micro",device="cuda:0")
    first = pilot(*args,1,temp/"resume",temp/"first.json",profile="micro",device="cuda:0")
    resumed = pilot(*args,2,temp/"resume",temp/"resumed.json",profile="micro",device="cuda:0",resume=True)
    for a,b in zip(cpu["updates"],direct["updates"]):
        assert a["indices"] == b["indices"] and abs(a["loss"] - b["loss"]) <= 0.02
    assert first["updates"] + resumed["updates"] == direct["updates"]
    def component_hashes(record, root):
        path = root/record["lineage"]/"objects"/record["final_checkpoint_sha256"]/"manifest.json"
        return {row["name"]:row["sha256"] for row in json.loads(path.read_bytes())["components"]}
    x,y = component_hashes(direct,temp/"direct"), component_hashes(resumed,temp/"resume")
    for name in ("model.bin","optimizer.bin","rng.bin","scheduler.json","cursor.json","ledger.json"):
        assert x[name] == y[name], name
    assert direct["evaluations"][-1]["predictions_sha256"] == resumed["evaluations"][-1]["predictions_sha256"]
    canary = {"source":SOURCE,"gpu":gpu,"torch":str(torch.__version__),"cuda":torch.version.cuda,
              "direct":direct,"resumed":resumed,"cpu":cpu,"verified_components":x}
CANARY_PASSED = True
print("CPU/CUDA loss parity and exact same-GPU resume passed.")

In [ ]:
if EXECUTE_CAMPAIGN:
    assert CANARY_PASSED
    from google.colab import drive
    drive.mount(str(MOUNT))
    DRIVE.mkdir(parents=True,exist_ok=True)
    if not (DRIVE/"canary.json").exists():
        (DRIVE/"canary.json").write_text(json.dumps(canary,indent=2))
    deadline = time.monotonic() + 75*60
    for seed in (7,19,41):
        for method in ("capacity-matched","context-only","associative","two-read"):
            lineage = f"expanded-{method}-join-v1-binding-{seed}"
            local_store = CheckpointStore(LOCAL/"checkpoints",lineage)
            mirrored = read_mirror_pointer(DRIVE/"mirror",lineage_id=lineage)
            if mirrored and local_store.latest_sha256() is None:
                materialize_local(DRIVE/"mirror",local_store)
            completed = []
            for boundary in (384,768,1152,1536):
                target = DRIVE/f"{method}-{seed}-{boundary}.json"
                if not target.exists():
                    break
                done = json.loads(target.read_bytes())
                assert done["source_commit"] == SOURCE and done["lineage"] == lineage
                assert done["evaluations"][-1]["update"] == boundary
                completed.append((boundary,done["final_checkpoint_sha256"]))
            expected = completed[-1][1] if completed else None
            assert mirrored == expected == local_store.latest_sha256(), "Reconcile an interrupted publication before resuming."
            for boundary in (384,768,1152,1536)[len(completed):]:
                remaining = deadline-time.monotonic()
                if remaining < 60:
                    raise TimeoutError("Session cap reached. Resume from the same Drive run ID.")
                free = int(subprocess.check_output(
                    ["nvidia-smi","--query-gpu=memory.free","--format=csv,noheader,nounits"],text=True))
                assert free >= 6000, free
                with tempfile.TemporaryDirectory(prefix="elendril-rung-",dir=LOCAL) as temp:
                    receipt = Path(temp)/f"{method}-{seed}-{boundary}.json"
                    try:
                        row = pilot(method,seed,1536,boundary,LOCAL/"checkpoints",receipt,
                                    profile="expanded",device="cuda:0",resume=expected is not None,
                                    timeout=min(900,int(remaining)))
                    except Exception:
                        log = receipt.with_suffix(".log")
                        if log.exists():
                            shutil.copyfile(log,DRIVE/f"{receipt.stem}-failed-{time.time_ns()}.log")
                        raise
                    expected = row["final_checkpoint_sha256"]
                    mirror_checkpoint(local_store,DRIVE/"mirror",checkpoint_sha256=expected)
                    target = DRIVE/receipt.name
                    pending = DRIVE/f".{receipt.name}-{time.time_ns()}.tmp"
                    shutil.copyfile(receipt,pending)
                    assert hashlib.sha256(pending.read_bytes()).digest() == hashlib.sha256(receipt.read_bytes()).digest()
                    os.replace(pending,target)
                    print(method,seed,boundary,row["evaluations"][-1]["metrics"]["query_swap_both_correct"],flush=True)
    print("Complete. Return the full Drive receipt set and canary evidence.")
else:
    print("Canary only. Enable EXECUTE_CAMPAIGN to run the expanded experiment.")